# **LangGraph Agent with Presidio PII Redaction & Sentiment-Driven Feedback**

## **Overview**
This notebook demonstrates how to build a stateful, multi-turn AI agent using **LangGraph** that dynamically adapts to user feedback while enforcing privacy guardrails. It integrates **Microsoft Presidio** with a custom recognizer for **Indian Phone Numbers** to automatically redact sensitive PII (names, emails, phone numbers, SSNs) from cloud telemetry in **LangSmith** without altering the agent's natural local conversation context.

---

**Key Features**
* **Privacy-First Tracing**: Intercepts and redacts sensitive entities (including Indian phone number formats) before uploading execution logs to LangSmith cloud traces.
* **Sentiment & Directive Classification**: Uses structured output parsing to evaluate user feedback as `positive`, `negative`, or `neutral` and extract specific behavioral instructions.
* **Dynamic System Prompt Injection**: Automatically modifies system prompt instructions on the fly based on stored sentiment and active directives.
* **State Persistence**: Employs in-memory checkpointers (`MemorySaver`) to retain conversation context and active feedback rules across multiple turns.

In [2]:

!pip install -qU langgraph langchain-openai langchain-core langsmith presidio-analyzer presidio-anonymizer spacy

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
crewai 0.28.8 requires langchain<0.2.0,>=0.1.10, but you have langchain 0.2.17 which is incompatible.
crewai 0.28.8 requires openai<2.0.0,>=1.13.3, but you have openai 3.3.1 which is incompatible.
crewai-tools 0.1.6 requires langchain<0.2.0,>=0.1.4, but you have langchain 0.2.17 which is incompatible.
crewai-tools 0.1.6 requires openai<2.0.0,>=1.12.0, but you have openai 3.3.1 which is incompatible.
embedchain 0.1.113 requires langchain<0.2.0,>=0.1.4, but you have langchain 0.2.17 which is incompatible.
embedchain 0.1.113 requires langchain-openai<0.2.0,>=0.1.7, but you have langchain-openai 1.6.0 which is incompatible.
google-adk 2.3.0 requires

In [1]:
import warnings
warnings.filterwarnings("ignore")

## **Import Core Modules**

We import the necessary components:
* **Pydantic**: To define structured data models for feedback extraction.
* **LangChain & LangGraph**: Components for messages, system prompts, graph edges, memory checkpointers, and client tracers.
* **Presidio Engine**: Core classes for defining custom regular expression patterns and running PII anonymization passes.

In [2]:
import os
from typing import Optional, Literal
from pydantic import BaseModel, Field
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

from langsmith import Client
from langsmith.anonymizer import create_anonymizer
from langchain_core.tracers.langchain import LangChainTracer

import utils

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = "a"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Feedback Agent with Guardrails Demo"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

## **Presidio PII Engine with Indian Phone Number Support**

Here, we set up Microsoft Presidio for anonymizing sensitive data:
1. **Custom Regex Pattern**: Defines a pattern matching Indian mobile and landline numbers (formats like `+91-XXXXX-XXXXX`, `0XXXXXXXXXX`, or standard 10-digit mobile numbers).
2. **Recognizer Registration**: Adds the custom `IN_PHONE` recognizer to Presidio's active registry alongside default recognizers.
3. **Anonymizer Function**: Defines `presidio_anonymize_func()`, which sanitizes `PERSON`, `EMAIL_ADDRESS`, `US_SSN`, `PHONE_NUMBER`, and `IN_PHONE` entities.

In [4]:
# --- 1. PRESIDIO PII REDACTION ENGINE ---
indian_phone_pattern = Pattern(
    name="indian_phone_regex",
    regex=r"(?:(?:\+?91[\s\-\.]?)|0)?[6-9]\d{4}[\s\-\.]?\d{5}\b",
    score=0.85
)

# 2. Create Custom Recognizer for IN_PHONE
indian_phone_recognizer = PatternRecognizer(
    supported_entity="IN_PHONE",
    patterns=[indian_phone_pattern],
    context=["phone", "call", "mobile", "whatsapp", "number", "contact"]
)
analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(indian_phone_recognizer)
anonymizer = AnonymizerEngine()


def presidio_anonymize_func(text: str, path: list = None) -> str:
    """Anonymizes PERSON, EMAIL_ADDRESS, US_SSN, standard PHONE_NUMBER, and IN_PHONE."""
    if not isinstance(text, str) or not text.strip():
        return text

    results = analyzer.analyze(
        text=text,
        entities=["PERSON", "EMAIL_ADDRESS", "US_SSN", "PHONE_NUMBER", "IN_PHONE"],
        language="en"
    )
    return anonymizer.anonymize(text=text, analyzer_results=results).text

## **Configure Redacted LangSmith Tracer**

To prevent PII from leaking into cloud tracing logs without breaking local LLM conversation context:
* We pass our `presidio_anonymize_func` into LangSmith's `create_anonymizer()`.
* We wrap it inside a custom `LangChainTracer(client=ls_client)`.
* When attached to graph calls, local execution receives raw text (so the model knows names), but telemetry uploaded to LangSmith is automatically sanitized.

In [5]:
# --- 3. CONFIGURE LANGSMITH CLIENT & TRACER CALLBACK ---
# This client sanitizes all trace payloads before uploading to LangSmith cloud
ls_client = Client(anonymizer=create_anonymizer(presidio_anonymize_func))
custom_tracer = LangChainTracer(client=ls_client)

## **Define Structured Feedback & Agent State Schemas**

We create Pydantic schemas to structure user feedback and track conversation state:
* `FeedbackItem`: Stores sentiment (`positive`, `negative`, `neutral`) and the extracted directive string.
* `ClassifierResult`: Output schema for the classification step to determine if feedback was provided.
* `AgentState`: Inherits from `MessagesState` to persist graph message history and store `user_feedback`.

In [6]:
# --- 4. STRUCTURED FEEDBACK & SCHEMAS ---
class FeedbackItem(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Sentiment: 'positive' (praise), 'negative' (complaint/issue), or 'neutral' (formatting directive)."
    )
    feedback_text: str = Field(
        description="The extracted rule, adjustment, or praise directive."
    )

class ClassifierResult(BaseModel):
    is_feedback: bool = Field(description="True if input contains feedback/behavior instructions, False otherwise.")
    feedback: Optional[FeedbackItem] = Field(default=None, description="The feedback item if is_feedback is True.")

class AgentState(MessagesState):
    user_feedback: Optional[FeedbackItem] = None

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## **Node: Feedback Classifier**

This node analyzes the latest user turn using OpenAI's Structured Output API (`with_structured_output`):
* Detects whether the turn contains explicit feedback or behavioral directives.
* If detected, extracts the sentiment (`positive`, `negative`, `neutral`) and updates `user_feedback` in the graph state.

In [8]:
# --- 5. GRAPH NODES ---
def feedback_classifier(state: AgentState):
    """Classifies user input for sentiment and feedback directives using structured outputs."""
    if not state["messages"]:
        return {}

    last_turn = state["messages"][-1].content
    structured_llm = llm.with_structured_output(ClassifierResult)

    prompt = [
        SystemMessage(content=(
            "Analyze if the user is giving feedback (positive praise, negative criticism, or behavioral instructions) "
            "regarding your responses or tone. If YES, set is_feedback to True and populate sentiment and feedback_text. "
            "If NO, set is_feedback to False."
        )),
        HumanMessage(content=last_turn)
    ]

    result = structured_llm.invoke(prompt)

    if result.is_feedback and result.feedback:
        return {"user_feedback": result.feedback}
    return {}

## **Node: Call Model (Agent Response Node)**

This node constructs the system prompt dynamically based on stored state:
* **Negative Feedback**: Injects a `CRITICAL ADJUSTMENT` directive into the prompt.
* **Positive Feedback**: Injects a `STYLE DIRECTIVE` to reinforce behavior.
* Generates the agent's response using the updated system instructions and message history.

In [9]:
def call_model(state: AgentState):
    """Constructs system prompt based on extracted sentiment & directive."""
    base_instructions = """You are a helpful customer service assistant.
    Your response should be precise and should not exceed 50 words"""

    # Retrieve the currently active feedback object from your agent's
    # state graph. Looks up the key "user_feedback" inside your LangGraph state dictionary.
    # If feedback was saved in a previous turn (like a user directive or sentiment object),
    # it fetches it; if no feedback exists yet, it returns.
    # Optional[FeedbackItem]: A Python type hint stating that the variable fb will either hold
    # a structured FeedbackItem object (containing sentiment and feedback_text) or None.

    fb: Optional[FeedbackItem] = state.get("user_feedback")

    if fb:
        if fb.sentiment == "negative":
            directive = (
                f"\n\nCRITICAL ADJUSTMENT (USER DISSATISFIED):\n"
                f"- Sentiment: Negative\n"
                f"- Required Change: {fb.feedback_text}"
            )
        elif fb.sentiment == "positive":
            directive = (
                f"\n\nSTYLE DIRECTIVE (USER PRAISED THIS ASPECT):\n"
                f"- Sentiment: Positive\n"
                f"- Keep Doing: {fb.feedback_text}"
            )
        else:
            directive = f"\n\nUSER DIRECTIVE:\n- {fb.feedback_text}"

        system_content = f"{base_instructions}{directive}"
    else:
        system_content = base_instructions

    messages = [SystemMessage(content=system_content)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

# **Assemble and Compile the LangGraph**

We assemble our application workflow:
1. **Nodes**: Add `classify_feedback` and `agent` nodes.
2. **Edges**: Route `START -> classify_feedback -> agent -> END`.
3. **Checkpointer**: Attach `MemorySaver()` to retain conversation history across turns per `thread_id`.

In [10]:
# --- 6. BUILD & COMPILE GRAPH ---
builder = StateGraph(AgentState)

builder.add_node("classify_feedback", feedback_classifier)
builder.add_node("agent", call_model)

builder.add_edge(START, "classify_feedback")
builder.add_edge("classify_feedback", "agent")
builder.add_edge("agent", END)

checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

## **Interactive Terminal Loop Definition**

Defines `run_cli_session()` to run an interactive prompt loop:
* **Tracing Callback**: Attaches `custom_tracer` to `config["callbacks"]` so trace payloads are redacted before being uploaded.
* **`/feedback` Command**: Inspects current active feedback state directly from the checkpointer.

In [11]:
# --- 7. INTERACTIVE CLI SESSION ---
def run_cli_session():
    print("==========================================================")
    print(" LangGraph Agent: Your helpful customer support agent")
    print(" Commands:")
    print("   • Type 'exit' or 'quit' to end session")
    print("   • Type '/feedback' to inspect active sentiment & rule")
    print("==========================================================\n")

    thread_id = "cli_session_1"

    # ATTACH CUSTOM TRACER TO CONFIG
    # This forces LangGraph to route all trace logging through our anonymizing client
    config = {
        "configurable": {"thread_id": thread_id},
        "callbacks": [custom_tracer]
    }

    while True:
        try:
            user_input = input("\nUser > ").strip()

            if not user_input:
                continue
            if user_input.lower() in ["exit", "quit"]:
                print("Ending conversation session. Goodbye!")
                break

            if user_input.lower() == "/feedback":
                current_state = app.get_state(config)
                fb: Optional[FeedbackItem] = current_state.values.get("user_feedback")

                if fb:
                    print(f"\n[DEBUG] Active Feedback State:")
                    print(f"  • Sentiment : {fb.sentiment.upper()}")
                    print(f"  • Directive : \"{fb.feedback_text}\"")
                else:
                    print("\n[DEBUG] No feedback currently active in state.")
                continue

            # Pass raw user input to app.invoke().
            # 1. Local LLM receives "Abc and responds naturally with "Hi Abc..."
            # 2. 'custom_tracer' intercepts trace payload and applies Presidio BEFORE sending to LangSmith.
            input_payload = {"messages": [HumanMessage(content=user_input)]}
            output = app.invoke(input_payload, config)
            agent_response = output["messages"][-1].content

            print(f"\nAgent > {agent_response}")

        except (KeyboardInterrupt, EOFError):
            print("\nSession interrupted. Exiting.")
            break

In [12]:
run_cli_session()

 LangGraph Agent: Your helpful customer support agent
 Commands:
   • Type 'exit' or 'quit' to end session
   • Type '/feedback' to inspect active sentiment & rule


Agent > Hi Ankur, please try the following steps: check the power connection, remove the battery (if possible), hold the power button for 15 seconds, then reconnect and try powering on again. If it still doesn't work, consider seeking professional help.

Agent > I apologize for the inconvenience. Could you provide more details about your laptop model and any indicators (like lights or sounds) when you try to power it on? This will help me assist you better.


Deserializing unregistered type __main__.FeedbackItem from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'FeedbackItem')]



Agent > Goodbye! If you need further assistance, feel free to reach out.

Session interrupted. Exiting.


In [14]:
# View feedback and message history
config = {"configurable": {"thread_id": "cli_session_1"}}

# Retrieve current state snapshot
state_snapshot = app.get_state(config)

# 1. View all active state variables (dictionary)
print("=== CURRENT STATE VALUES ===")
for key, value in state_snapshot.values.items():
    print(f"{key}: {value}\n")

=== CURRENT STATE VALUES ===
messages: [HumanMessage(content='Hi, My name is Ankur and my lapto is not powering on', additional_kwargs={}, response_metadata={}, id='9f30e3ad-2301-4ded-87dc-c991c2ca0831'), AIMessage(content="Hi Ankur, please try the following steps: check the power connection, remove the battery (if possible), hold the power button for 15 seconds, then reconnect and try powering on again. If it still doesn't work, consider seeking professional help.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 47, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'f

In [ ]:
# 2. View message history list
print("=== CONVERSATION HISTORY ===")
for msg in state_snapshot.values.get("messages", []):
    print(f"{msg.type.upper()}: {msg.content}")